# Training Basics: solutions

**Author:** Nassim Razaaly ([nassim.razaaly@ensma.fr](mailto:nassim.razaaly@ensma.fr))

Solutions of the exercises of the notebook *Training Basics*. Each part starts with a hidden cell that loads the tools of the main notebook: run it first. The numbers quoted in the text are indicative: yours may differ a little (random initialization and data).

## 1. A 1D function

In [ ]:
# @title Setup: tools of the main notebook (run me)
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cbook
from IPython.display import display
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
loss_fn = nn.MSELoss()

class MLP(nn.Module):
    def __init__(self, layers, activation, last_activation=None):
        super().__init__()
        blocks = []
        for n_in, n_out in zip(layers[:-2], layers[1:-1]):
            blocks += [nn.Linear(n_in, n_out), activation()]
        blocks.append(nn.Linear(layers[-2], layers[-1]))
        if last_activation is not None:
            blocks.append(last_activation())
        self.net = nn.Sequential(*blocks)
    def forward(self, x):
        return self.net(x)

def f(x):
    return torch.sin(3 * x) + 0.3 * x**2 - 1.5 * torch.exp(-10 * (x - 1) ** 2)

def generate_data(N, noise, train=0.6, val=0.2, test=0.2, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.rand(N, 1, generator=g) * 6 - 3
    y = f(x) + noise * torch.randn(N, 1, generator=g)
    perm = torch.randperm(N, generator=g)
    n_tr, n_va = int(train * N), int(val * N)
    idx = {"train": perm[:n_tr], "val": perm[n_tr:n_tr + n_va], "test": perm[n_tr + n_va:]}
    tr = idx["train"]
    norm = {"x": (x[tr].mean(), x[tr].std()), "y": (y[tr].mean(), y[tr].std())}
    data = {k: (((x[i] - norm["x"][0]) / norm["x"][1]).to(device), ((y[i] - norm["y"][0]) / norm["y"][1]).to(device))
            for k, i in idx.items()}
    return data, norm

def train(model, optimizer, data, n_epochs, n_batch, plot_every=100, scheduler=None, snapshot=None, title=""):
    X, Y = data["train"]
    X_val, Y_val = data["val"]
    hist = {"train": [], "val": [], "snapshots": []}
    fig, ax = plt.subplots(figsize=(6, 3)); live = display(fig, display_id=True); plt.close(fig)
    for epoch in tqdm(range(n_epochs)):
        model.train()
        for idx in torch.randperm(len(X), device=X.device).chunk(n_batch):
            optimizer.zero_grad()
            loss = loss_fn(model(X[idx]), Y[idx])
            loss.backward()
            optimizer.step()
        if scheduler is not None:
            scheduler.step()
        model.eval()
        with torch.no_grad():
            hist["train"].append(loss_fn(model(X), Y).item())
            hist["val"].append(loss_fn(model(X_val), Y_val).item())
        if (epoch + 1) % plot_every == 0 or epoch == n_epochs - 1:
            if snapshot is not None:
                hist["snapshots"].append(snapshot())
            ax.clear()
            ax.semilogy(hist["train"], label="train"); ax.semilogy(hist["val"], "--", label="validation")
            ax.set_xlabel("epoch"); ax.set_ylabel("MSE"); ax.set_title(title); ax.legend()
            live.update(fig)
    return hist

def predict(model, x, norm):
    model.eval()
    with torch.no_grad():
        y = model(((x - norm["x"][0]) / norm["x"][1]).to(device)).cpu()
    return norm["y"][0] + norm["y"][1] * y

x_grid = torch.linspace(-3, 3, 500)[:, None]

# ---- helpers of the solutions
def fit(layers, activation, data, lr, n_epochs, n_batch, cosine=False, title="", seed=0, stop=None, show_lr=False):
    """Build and train an MLP (Adam, optional cosine schedule); stop: end the training earlier, same schedule;
    show_lr: plot the learning rate along the training."""
    torch.manual_seed(seed)
    model = MLP(layers, activation).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs) if cosine else None
    n_run = n_epochs if stop is None else stop
    t0 = time.perf_counter()
    hist = train(model, optimizer, data, n_run, n_batch, plot_every=n_run, scheduler=scheduler, title=title)
    hist["time"] = time.perf_counter() - t0
    print(f"{title}: training time {hist['time']:.1f} s" + (f" | {lr_info(lr, n_epochs, n_run)}" if cosine else ""))
    if cosine and show_lr:
        plot_lr(lr, n_epochs, title)
    return model, hist

def mse(model, data, norm, name):
    """MSE on one set, in the original units of y."""
    model.eval()
    with torch.no_grad():
        return loss_fn(model(data[name][0]), data[name][1]).item() * norm["y"][1].item() ** 2

def plot_fits(models, data, norm, title):
    """Predictions of several models ({label: model}), true function and training points."""
    X, Y = data["train"]
    plt.figure(figsize=(8, 3.5))
    plt.scatter(norm["x"][0] + norm["x"][1] * X.cpu(), norm["y"][0] + norm["y"][1] * Y.cpu(),
                s=8, color="gray", alpha=0.5, label="training points")
    plt.plot(x_grid, f(x_grid), "k", lw=2, label="true function")
    for label, m in models.items():
        plt.plot(x_grid, predict(m, x_grid, norm), lw=1.5, label=label)
    plt.xlabel("x"); plt.ylabel("y"); plt.title(title); plt.legend(fontsize=8); plt.show()

def lr_curve(lr, n_epochs, cosine):
    """Learning rate at each epoch, constant or with the cosine schedule."""
    opt = torch.optim.Adam([nn.Parameter(torch.zeros(1))], lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs) if cosine else None
    lrs = []
    for _ in range(n_epochs):
        lrs.append(opt.param_groups[0]["lr"])
        if sched is not None:
            opt.step(); sched.step()
    return lrs

def lr_info(lr, n_epochs, n_run=None):
    lrs = lr_curve(lr, n_epochs, cosine=True)[:n_run or n_epochs]       # n_run: epochs actually run
    return f"cosine schedule: lr from {lr:g} down to {lrs[-1]:.1e} (lowest lr, last epoch)"

def plot_lr(lr, n_epochs, title=""):
    """Learning rate of the cosine schedule along the training."""
    lrs = lr_curve(lr, n_epochs, cosine=True)
    plt.figure(figsize=(5, 2.5))
    plt.plot(np.arange(1, n_epochs + 1), lrs)
    plt.xlabel("epoch"); plt.ylabel("learning rate")
    plt.title(f"{title}\ncosine schedule: lr from {lr:g} to {lrs[-1]:.1e}", fontsize=9)
    plt.show()

print("device:", device)

### Exercise 1

We study the learning rate and the mini-batches with little noise (`noise = 0.01`), where their effect is easier to see, then apply the best settings to the data of section 1.2.

**Learning rate** (2 hidden layers of 64 neurons, full batch, 3000 epochs):

In [ ]:
data_01, norm_01 = generate_data(N=500, noise=0.01)
lr_runs = {"lr = 1e-4 (too small)": (1e-4, False), "lr = 1e-1 (too large)": (1e-1, False),
           "lr = 1e-2": (1e-2, False), "lr = 1e-2 + cosine": (1e-2, True)}
models_lr, hists_lr = {}, {}
for name, (lr, cos) in lr_runs.items():
    models_lr[name], hists_lr[name] = fit([1, 64, 64, 1], nn.Tanh, data_01, lr=lr, n_epochs=3000, n_batch=1, cosine=cos, title=name)
    print(f"{name:22s}: val MSE = {mse(models_lr[name], data_01, norm_01, 'val'):.1e}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for name, (lr, cos) in lr_runs.items():
    ax[0].semilogy(hists_lr[name]["val"], label=name)
    lrs = lr_curve(lr, 3000, cos)
    ax[1].semilogy(lrs, label=f"{name} (lowest lr = {min(lrs):.0e})")
ax[0].set(xlabel="epoch", ylabel="validation MSE (normalized)", title="Validation loss")
ax[1].set(xlabel="epoch", ylabel="learning rate", title="Learning rate")
ax[0].legend(fontsize=8); ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()

* **Too small** (`1e-4`): tiny steps, the loss is still high after 3000 epochs.
* **Too large** (`1e-1`): the steps jump over the minimum; the loss is unstable, with large spikes, and ends high.
* **Well chosen** (`1e-2`): the loss quickly reaches about $10^{-4}$ (the noise level, `noise`$^2$), but keeps oscillating.
* **Well chosen + cosine schedule**: large steps at the beginning, small at the end: fast progress, then a smooth convergence. The best choice.

**Cosine schedule** (right panel, red): $\text{lr}(t) = \text{lr}_0 \, \dfrac{1 + \cos(\pi t / T)}{2}$, from $\text{lr}_0$ at the first epoch to 0 at the last one ($T$ = `n_epochs`; the lowest value can be raised with `CosineAnnealingLR(..., eta_min=...)`). It decreases slowly at the beginning, fastest at mid-training, and slowly again at the end; the lowest lr actually used, at the last epoch, is about $10^{-9}$ here.

**Mini-batches** (same model, `lr = 1e-3`, 500 epochs):

In [ ]:
nb_runs = {"n_batch = 1": (1, False), "n_batch = 10": (10, False), "n_batch = 50": (50, False), "n_batch = 50 + cosine": (50, True)}
hists_nb = {}
for name, (nb, cos) in nb_runs.items():
    _, hists_nb[name] = fit([1, 64, 64, 1], nn.Tanh, data_01, lr=1e-3, n_epochs=500, n_batch=nb, cosine=cos, title=name)

plt.figure(figsize=(7, 3.5))
for name, h in hists_nb.items():
    plt.semilogy(h["val"], label=f"{name} ({h['time']:.0f} s)")
plt.xlabel("epoch"); plt.ylabel("validation MSE (normalized)"); plt.title("Effect of the mini-batches (lr = 1e-3)")
plt.legend(fontsize=8); plt.show()

* More mini-batches = more updates per epoch: the loss decreases much faster per epoch, but each epoch costs more (times in the legend).
* With small mini-batches (6 points for `n_batch = 50`), the gradient is noisy, and so is the loss at the end; the cosine schedule removes this noise.

**Best model on the data of section 1.2** (`noise = 0.2`, $N = 500$): 2 hidden layers of 64 neurons, `lr = 1e-2` with cosine schedule, full batch, 2000 epochs. Then the same with more data ($N = 2000$), our final model (test MSE computed once).

In [ ]:
data, norm = generate_data(N=500, noise=0.2)
model_12, _ = fit([1, 32, 1], nn.Tanh, data, lr=1e-3, n_epochs=2000, n_batch=2, title="section 1.2 settings")
model_best, _ = fit([1, 64, 64, 1], nn.Tanh, data, lr=1e-2, n_epochs=2000, n_batch=1, cosine=True, title="best settings", show_lr=True)
print(f"section 1.2 settings: val MSE = {mse(model_12, data, norm, 'val'):.4f}")
print(f"best settings       : val MSE = {mse(model_best, data, norm, 'val'):.4f}")
plot_fits({"section 1.2 settings": model_12, "best settings": model_best}, data, norm, "noise = 0.2, N = 500")

data_2000, norm_2000 = generate_data(N=2000, noise=0.2)
model_2000, _ = fit([1, 64, 64, 1], nn.Tanh, data_2000, lr=1e-2, n_epochs=2000, n_batch=1, cosine=True, title="best settings, N = 2000")
print(f"N = 2000: val MSE = {mse(model_2000, data_2000, norm_2000, 'val'):.4f} | test MSE = {mse(model_2000, data_2000, norm_2000, 'test'):.4f}")

The section 1.2 model is under-trained (val MSE about 0.2); the best settings reach about 0.043, and 0.041 with $N = 2000$, close to the lowest possible value, `noise`$^2 = 0.04$ (exercise 2).

### Exercise 2

Same data as in section 1.2 ($N = 500$), with a very small noise, `noise = 0.01`. We train two models and compare their loss curves, validation MSE and predictions:

1. the **section 1.2 settings**: 1 hidden layer of 32 neurons, `lr = 1e-3`, `n_batch = 2`, 2000 epochs;
2. the **best settings of exercise 1**: 2 hidden layers of 64 neurons, `lr = 1e-2` with cosine schedule, full batch, 3000 epochs.

In [ ]:
model_01_12, _ = fit([1, 32, 1], nn.Tanh, data_01, lr=1e-3, n_epochs=2000, n_batch=2, title="noise = 0.01: section 1.2 settings")

In [ ]:
model_01, _ = fit([1, 64, 64, 1], nn.Tanh, data_01, lr=1e-2, n_epochs=3000, n_batch=1, cosine=True, title="noise = 0.01: best settings", show_lr=True)
print(f"section 1.2 settings: val MSE = {mse(model_01_12, data_01, norm_01, 'val'):.5f}")
print(f"best settings       : val MSE = {mse(model_01, data_01, norm_01, 'val'):.5f}   (noise^2 = 0.0001)")
plot_fits({"section 1.2 settings": model_01_12, "best settings": model_01}, data_01, norm_01, "noise = 0.01, N = 500")

* The MSE is measured against **noisy** values $y = f(x) + \varepsilon$: even a perfect model has an MSE equal to `noise`$^2$, the lowest reachable value (0.04 for `noise = 0.2`, $10^{-4}$ for `noise = 0.01`).
* **Section 1.2 settings**: the loss decreases slowly and is still far from its minimum after 2000 epochs (val MSE about 0.17): the model is under-trained, its curve misses the details of $f$. With `noise = 0.2` the same settings gave about 0.2 (exercise 1): the error comes from the training, not from the noise.
* **Best settings**: the loss drops to about $1.4 \cdot 10^{-4}$, almost the noise level, and the curve is on top of the true function. With `noise = 0.2`, the same settings stop at about 0.043, also close to the noise level (0.04).
* Conclusion: with noisy data, the noise limits the MSE and a good model cannot gain much; with clean data, the error comes only from the model and its training, so the hyperparameters matter much more.

### Exercise 3

We make a large model overfit few points: $N = 100$ (60 training points), 3 hidden layers of 128 neurons, trained long and well (4000 epochs, `lr = 1e-2` with cosine schedule). Then we train it again with **early stopping**, and compare the two predictions.

In [ ]:
data_100, norm_100 = generate_data(N=100, noise=0.2)
settings_over = dict(layers=[1, 128, 128, 128, 1], activation=nn.SiLU, data=data_100, lr=1e-2, n_epochs=4000, n_batch=1, cosine=True)
model_over, hist_over = fit(**settings_over, title="large model, 60 training points", show_lr=True)
best_epoch = int(np.argmin(hist_over["val"])) + 1
model_early, _ = fit(**settings_over, stop=best_epoch, title=f"early stopping at epoch {best_epoch}")

print(f"4000 epochs   : train MSE = {mse(model_over, data_100, norm_100, 'train'):.4f} | val MSE = {mse(model_over, data_100, norm_100, 'val'):.4f}")
print(f"early stopping: train MSE = {mse(model_early, data_100, norm_100, 'train'):.4f} | val MSE = {mse(model_early, data_100, norm_100, 'val'):.4f}")
plot_fits({"4000 epochs (overfitted)": model_over, f"early stopping ({best_epoch} epochs)": model_early},
          data_100, norm_100, "Overfitting, 60 training points")

* The training MSE goes below the noise level (0.04): the model fits the **noise** of the 60 points. The validation loss reaches a minimum, then increases; the curve wiggles between the training points.
* **Early stopping**: keep the model of the epoch with the lowest validation loss (here: same training, stopped at `best_epoch`). The curve is smoother and the validation MSE lower.
* Other solutions: more data, a smaller model, or a regularization (e.g. `torch.optim.AdamW`, with weight decay).

### Exercise 4

Same data (`noise = 0.2`), model and training as the best settings of exercise 1; only the activation changes: ReLU, Tanh, SiLU.

In [ ]:
models_act = {}
for act in [nn.ReLU, nn.Tanh, nn.SiLU]:
    models_act[act.__name__], _ = fit([1, 64, 64, 1], act, data, lr=1e-2, n_epochs=2000, n_batch=1, cosine=True, title=act.__name__)
    print(f"{act.__name__:5s}: val MSE = {mse(models_act[act.__name__], data, norm, 'val'):.4f}")
plot_fits(models_act, data, norm, "Activation functions (noise = 0.2)")

Tanh (about 0.043) and SiLU (about 0.045) are clearly better than ReLU (about 0.067). With ReLU, the MLP is **piecewise linear**: it approximates the smooth $f$ by straight segments with kinks, and needs many neurons for the curved parts. Tanh and SiLU are smooth, like $f$.

### To go further: a 2D function

$f_2(x_1, x_2) = \sin(\pi x_1)\cos(\pi x_2)$ on $[-1, 1]^2$. Inputs in $[-1, 1]$ and outputs of order 1: no normalization needed. `generate_data_2d(N, noise)` draws the points and splits them (60 / 20 / 20 %). We plot the data ($N = 300$, `noise = 0.01`), train a small MLP, then compare the true function, the prediction and the error.

In [ ]:
def f2(x):                                                  # (N, 2) -> (N, 1)
    return torch.sin(torch.pi * x[:, :1]) * torch.cos(torch.pi * x[:, 1:])

def generate_data_2d(N, noise, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.rand(N, 2, generator=g) * 2 - 1
    y = f2(x) + noise * torch.randn(N, 1, generator=g)
    n_tr, n_va = int(0.6 * N), int(0.2 * N)
    return {"train": (x[:n_tr].to(device), y[:n_tr].to(device)),
            "val": (x[n_tr:n_tr + n_va].to(device), y[n_tr:n_tr + n_va].to(device)),
            "test": (x[n_tr + n_va:].to(device), y[n_tr + n_va:].to(device))}

data_2d = generate_data_2d(N=300, noise=0.01)

fig, ax = plt.subplots(figsize=(5, 4))
for name, marker in [("train", "o"), ("val", "s"), ("test", "^")]:
    X2, Y2 = data_2d[name]
    sc = ax.scatter(X2[:, 1].cpu(), X2[:, 0].cpu(), c=Y2[:, 0].cpu(), cmap="RdBu", vmin=-1, vmax=1, marker=marker,
                    s=20, edgecolors="k", linewidths=0.3, label=f"{name} ({len(X2)} points)")
ax.set(xlabel="$x_2$", ylabel="$x_1$", title="Data: N = 300, noise = 0.01")
fig.colorbar(sc, label="$y$"); ax.legend(fontsize=8, loc="upper right"); plt.show()

In [ ]:
model_2d, hist_2d = fit([2, 32, 32, 1], nn.Tanh, data_2d, lr=1e-2, n_epochs=3000, n_batch=1, cosine=True, title="2D function", show_lr=True)
print(f"train MSE = {hist_2d['train'][-1]:.1e} | val MSE = {hist_2d['val'][-1]:.1e}")

t = torch.linspace(-1, 1, 100)
grid = torch.stack(torch.meshgrid(t, t, indexing="ij"), dim=-1).reshape(-1, 2)
with torch.no_grad():
    pred = model_2d(grid.to(device)).cpu().reshape(100, 100)
true = f2(grid).reshape(100, 100)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
for a, z, title in zip(ax, [true, pred, pred - true], ["true function $f_2$", "MLP", "error (MLP - true)"]):
    im = a.imshow(z, extent=[-1, 1, -1, 1], origin="lower", cmap="RdBu", vmin=-1, vmax=1)
    a.set(title=title, xlabel="$x_2$", ylabel="$x_1$")
fig.colorbar(im, ax=ax, label="value (same scale for the 3 maps)"); plt.show()

With 180 training points, a small MLP learns $f_2$ in a few seconds: the training MSE reaches the noise level ($10^{-4}$), the validation MSE is a few $10^{-4}$, and the error map is almost blank at the scale of the function (the largest errors are at the corners, where there are few points).

## 2. Images

In [ ]:
# @title Setup: tools of the main notebook (run me)
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def load_image(path, size=128, gray=False):
    raw = plt.imread(path)
    im = torch.tensor(raw, dtype=torch.float32)
    if raw.dtype == np.uint8:
        im = im / 255
    if im.ndim == 2:
        im = im[..., None]
    if im.shape[2] == 4:
        im = im[..., :3] * im[..., 3:] + 1 - im[..., 3:]
    H, W = im.shape[:2]; s = min(H, W)
    im = im[(H - s) // 2:(H - s) // 2 + s, (W - s) // 2:(W - s) // 2 + s]
    im = F.interpolate(im.permute(2, 0, 1)[None], size=(size, size), mode="area")[0].permute(1, 2, 0)
    if gray:
        im = im.mean(dim=2, keepdim=True)
    return im.squeeze(2)

def show_images(images, titles, size=3):
    fig, ax = plt.subplots(1, len(images), figsize=(size * len(images), size))
    for a, im, t in zip(np.atleast_1d(ax), images, titles):
        a.imshow(np.clip(im, 0, 1), cmap="gray", vmin=0, vmax=1); a.set_title(t, fontsize=10); a.axis("off")
    plt.tight_layout(); plt.show()

def image_data(img, train=0.6, val=0.2, seed=0):
    n = img.shape[0]
    t = torch.linspace(-1, 1, n)
    yy, xx = torch.meshgrid(t, t, indexing="ij")
    X = torch.stack([xx.flatten(), yy.flatten()], dim=1)
    Y = img.reshape(n * n, -1)
    perm = torch.randperm(n * n, generator=torch.Generator().manual_seed(seed))
    n_tr, n_va = int(train * n * n), int(val * n * n)
    idx = {"train": perm[:n_tr], "val": perm[n_tr:n_tr + n_va], "test": perm[n_tr + n_va:]}
    return {k: (X[i].to(device), Y[i].to(device)) for k, i in idx.items()}, X.to(device)

def predict_image(model, X_all, shape):
    model.eval()
    with torch.no_grad():
        return model(X_all).cpu().numpy().reshape(shape)

def psnr(mse):
    return 10 * np.log10(1 / mse)

class FourierFeatures(nn.Module):
    def __init__(self, n_in, n_freq, sigma, trainable=False):
        super().__init__()
        self.B = nn.Parameter(sigma * torch.randn(n_freq, n_in), requires_grad=trainable)
    def forward(self, v):
        z = 2 * torch.pi * v @ self.B.T
        return torch.cat([torch.cos(z), torch.sin(z)], dim=1)

# ---- helpers of the solutions
def ff_model(n_freq, sigma, hidden, n_out=1, trainable=False):
    """Fourier features, then an MLP with the given hidden layers and a sigmoid output."""
    return nn.Sequential(FourierFeatures(2, n_freq, sigma, trainable),
                         MLP([2 * n_freq] + hidden + [n_out], nn.ReLU, nn.Sigmoid))

def fit_image(model, im, lr, n_epochs, n_batch, cosine=False, title="", plot_every=None, show_lr=False):
    """Train a model on an image (Adam, optional cosine schedule); returns the history, the model and the data."""
    data_im, X_im = image_data(im)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs) if cosine else None
    t0 = time.perf_counter()
    hist = train(model, optimizer, data_im, n_epochs, n_batch, plot_every=plot_every or n_epochs, scheduler=scheduler, title=title)
    hist.update(time=time.perf_counter() - t0, psnr_train=psnr(hist["train"][-1]), psnr_val=psnr(hist["val"][-1]),
                image=predict_image(model, X_im, im.shape), data=data_im)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{title} ({n_params} parameters): train PSNR = {hist['psnr_train']:.1f} dB | val PSNR = {hist['psnr_val']:.1f} dB | "
          f"training time {hist['time']:.0f} s" + (f" | {lr_info(lr, n_epochs)}" if cosine else ""))
    if cosine and show_lr:
        plot_lr(lr, n_epochs, title)
    return hist

def interp_psnr(im):
    """Reference without network: validation PSNR of a cubic interpolation from the training pixels."""
    from scipy.interpolate import griddata
    d, _ = image_data(im)
    pred = griddata(d["train"][0].cpu().numpy(), d["train"][1].cpu().numpy(), d["val"][0].cpu().numpy(), method="cubic")
    pred = np.nan_to_num(pred, nan=0.5)                     # pixels outside the hull of the training pixels
    return psnr(((pred - d["val"][1].cpu().numpy()) ** 2).mean())

def test_psnr(model, hist):
    model.eval()
    with torch.no_grad():
        return psnr(loss_fn(model(hist["data"]["test"][0]), hist["data"]["test"][1]).item())

img = load_image(cbook.get_sample_data("Minduka_Present_Blue_Pack.png", asfileobj=False), gray=True)
img_color = load_image(cbook.get_sample_data("grace_hopper.jpg", asfileobj=False), size=128)
show_images([img.numpy(), img_color.numpy()], ["drawing (gray)", "portrait (color)"])

### Exercise 1

**What we do.** Without Fourier features, all the MLPs we tried level off around 22 dB (the architecture matters most: 1 hidden layer of 64 neurons gives about 16 dB, 3 layers of 128 about 21 dB; a larger `lr` with cosine schedule adds about 0.5 dB), see also part 3, exercise 1. The best results need **Fourier features** (section 2.7, studied in part 3). Here we train the final models **to the end**:

1. **Reference without network:** cubic interpolation of the validation pixels from the training pixels.
2. **Two models**, with the same Fourier features and two sizes of MLP:
   * **Fourier features:** $(x, y) \mapsto$ 256 random frequencies with $\sigma = 1.5$, i.e. 512 inputs (cos and sin) for the MLP. $B$ is fixed (not trained). $\sigma$ was chosen on the validation set (part 3, and long trainings: $\sigma = 1, 1.5, 2, 3$ give about 30.9, 31.2, 30.8, 29.6 dB).
   * **MLP:** $512 \to 128 \to 128 \to 1$ (about 82 000 parameters) or $512 \to 256 \to 256 \to 1$ (about 197 000), ReLU, sigmoid output.
   * **Training:** Adam, 100 epochs; each epoch has 100 mini-batches of about 98 pixels, so 10 000 updates in total. **Learning rate:** cosine schedule, one value per epoch, from $10^{-3}$ at epoch 1 to $5 \cdot 10^{-4}$ at epoch 50 and about $2.5 \cdot 10^{-7}$ at epoch 100 (figure below).

This takes about 2 to 5 minutes on a Colab CPU for the two models (much less on a GPU).

In [ ]:
print(f"cubic interpolation (no network): val PSNR = {interp_psnr(img):.1f} dB")

n_freq, sigma, lr, n_epochs, n_batch = 256, 1.5, 1e-3, 100, 100
models_draw, hists_draw = {}, {}
for hidden in ([128, 128], [256, 256]):
    name = f"FF + MLP {hidden}"
    torch.manual_seed(0)
    models_draw[name] = ff_model(n_freq, sigma, hidden)
    hists_draw[name] = fit_image(models_draw[name], img, lr, n_epochs, n_batch, cosine=True, title=name, plot_every=10)
plot_lr(lr, n_epochs, "learning rate of the two trainings")

chosen = "FF + MLP [128, 128]"
print(f"chosen model ({chosen}): test PSNR = {test_psnr(models_draw[chosen], hists_draw[chosen]):.1f} dB")
show_images([img.numpy()] + [h["image"] for h in hists_draw.values()],
            ["original"] + [f"{k}\nval {h['psnr_val']:.1f} dB" for k, h in hists_draw.items()], size=4)

* Cubic interpolation gives about 26 dB; both models about **31 dB** on the validation pixels (about 32 to 33 dB on the test pixels): differences with the original are hard to see.
* The small MLP is as good as the large one (31.1 vs 31.2 dB) with 2.4 times fewer parameters and a shorter training. **Chosen solution: Fourier features (256 frequencies, $\sigma = 1.5$) + MLP $[128, 128]$, 100 epochs, cosine schedule from $10^{-3}$.**
* Can we do better? Not much: a deeper MLP ($3 \times 256$) gives 31.5 dB for twice the time; SiLU (29.3 dB), a larger learning rate ($3 \cdot 10^{-3}$: 31.2 dB), smaller or larger $\sigma$ do not help. The training PSNR is much higher (about 42 dB): the model reproduces the known pixels almost exactly, and the remaining error comes from the 40 % of pixels it must guess.

### Exercise 2

No new training: we look at the animation of section 2.4. The large uniform areas and the overall shape (low frequencies) appear in the first epochs; the edges and small details (high frequencies) come much later and stay blurry. This is the **spectral bias** of MLPs: they learn the low frequencies of a function first, and the high frequencies very slowly. Fourier features give the high frequencies directly as inputs.

### Exercise 3

**What we do.** Same approach on the color portrait, with 3 outputs $(r, g, b)$; reference: cubic interpolation. Chosen model, from long trainings compared on the validation set:

* Fourier features: 256 frequencies, $\sigma = 2$ ($\sigma = 1, 1.5, 2, 3$ give about 25.2, 25.7, 25.7, 25.6 dB);
* MLP $512 \to 256 \to 256 \to 3$ (about 198 000 parameters): here the larger MLP helps a little ($[128, 128]$: 25.5 dB), the photo has more details than the drawing;
* same training as exercise 1: Adam, 100 epochs of 100 mini-batches, cosine schedule from $10^{-3}$ to about 0.

This takes about 2 to 4 minutes on a Colab CPU.

In [ ]:
print(f"cubic interpolation (no network): val PSNR = {interp_psnr(img_color):.1f} dB")
torch.manual_seed(0)
model_color = ff_model(n_freq=256, sigma=2, hidden=[256, 256], n_out=3)
hist_color = fit_image(model_color, img_color, lr=1e-3, n_epochs=100, n_batch=100, cosine=True, title="portrait: FF + MLP [256, 256]", plot_every=10)
print(f"test PSNR = {test_psnr(model_color, hist_color):.1f} dB")
show_images([img_color.numpy(), hist_color["image"]], ["original", f"FF + MLP [256, 256]: val {hist_color['psnr_val']:.1f} dB"], size=4)

* About **26 dB** on the validation and test pixels, against 23 dB for cubic interpolation: the face, the glasses and the uniform are sharp.
* 33 dB is not reachable here: the portrait has fine textures (hair, medals, flag, JPEG noise) that cannot be predicted from the neighboring pixels. A larger model (MLP $3 \times 256$: 25.8 dB), SiLU (25.2 dB) or smaller mini-batches (25.2 dB) do not help: they only raise the training PSNR (about 38 dB), i.e. the model memorizes the known pixels. The limit comes from the 40 % of unknown pixels: with more training pixels (e.g. `image_data(img_color, train=0.9, val=0.05)`), the PSNR on the remaining pixels increases.

### Exercise 4

Upload your image (Colab: folder icon on the left), then run the cell. With `size=256` there are 4 times more pixels: each epoch takes 4 times longer, and finer details may need a larger $\sigma$ (choose it on the validation set).

In [ ]:
import os
path = "my_image.jpg"
if os.path.exists(path):
    my_img = load_image(path, size=128)
    torch.manual_seed(0)
    model_mine = ff_model(n_freq=256, sigma=2, hidden=[256, 256], n_out=1 if my_img.ndim == 2 else 3)
    hist_mine = fit_image(model_mine, my_img, lr=1e-3, n_epochs=100, n_batch=100, cosine=True, title="my image", plot_every=10)
    show_images([my_img.numpy(), hist_mine["image"]], ["original", f"val {hist_mine['psnr_val']:.1f} dB"], size=4)
else:
    print(f"upload an image named {path} first")

## 3. Fourier features (section 2.7)

**Common setup of this part:** short trainings on the drawing, to compare settings quickly (about 1 min in total).

* Model: Fourier features (`n_freq` frequencies of scale $\sigma$) $\to$ MLP $[2\,\cdot$ `n_freq`$, 128, 128, 1]$, ReLU, sigmoid output.
* Training: Adam, **constant** `lr = 1e-3`, 100 mini-batches per epoch, 20 or 30 epochs. The models are not trained to convergence: the PSNR are lower than in part 2.

### Exercise 1

We train 3 models for 30 epochs and compare their validation PSNR and images:

1. no Fourier features: MLP $[2, 128, 128, 1]$;
2. Fourier features, `n_freq = 128`, $\sigma = 10$ (cell of section 2.7);
3. the same with $\sigma = 3$.

In [ ]:
runs_ff = {}
for name, m in {"no Fourier features": MLP([2, 128, 128, 1], nn.ReLU, nn.Sigmoid),
                "FF, sigma = 10": ff_model(128, 10, [128, 128]),
                "FF, sigma = 3": ff_model(128, 3, [128, 128])}.items():
    torch.manual_seed(0)
    runs_ff[name] = fit_image(m, img, lr=1e-3, n_epochs=30, n_batch=100, title=name)
show_images([img.numpy()] + [h["image"] for h in runs_ff.values()],
            ["original"] + [f"{k}: val {h['psnr_val']:.1f} dB" for k, h in runs_ff.items()])

* Without Fourier features: about 18 dB, blurry.
* $\sigma = 3$: about 27 dB, sharp edges, in the same number of epochs.
* $\sigma = 10$ (the value of section 2.7): sharp edges but a noisy image, and a much lower validation PSNR (about 21 dB): the frequencies are too high, the model oscillates between the training pixels. $\sigma$ must be chosen.

### Exercises 2 and 3

**Exercise 2:** we choose $\sigma$ and `n_freq` on the validation set, in two steps (20 epochs each):

1. `n_freq = 64` and $\sigma \in \{1, 3, 10, 30\}$: we keep the $\sigma$ with the highest validation PSNR;
2. this $\sigma$, and `n_freq` $\in \{16, 256\}$.

Then we compute the test PSNR of the best setting, once. **Exercise 3:** training and validation curves for $\sigma = 3$ and $\sigma = 30$.

In [ ]:
models_ff, res_ff = {}, {}
def run_ff(n_freq, sigma):
    torch.manual_seed(0)
    models_ff[sigma, n_freq] = ff_model(n_freq, sigma, [128, 128])
    res_ff[sigma, n_freq] = fit_image(models_ff[sigma, n_freq], img, lr=1e-3, n_epochs=20, n_batch=100, title=f"sigma = {sigma}, n_freq = {n_freq}")

for sigma in [1, 3, 10, 30]:                                       # step 1
    run_ff(64, sigma)
best_sigma = max([1, 3, 10, 30], key=lambda s: res_ff[s, 64]["psnr_val"])
for n_freq in [16, 256]:                                           # step 2
    run_ff(n_freq, best_sigma)
best = max(res_ff, key=lambda k: res_ff[k]["psnr_val"])
print(f"best setting on the validation set: sigma = {best[0]}, n_freq = {best[1]} -> test PSNR = {test_psnr(models_ff[best], res_ff[best]):.1f} dB")
show_images([h["image"] for h in res_ff.values()], [f"sigma={s}, n_freq={nf}\nval {h['psnr_val']:.1f} dB" for (s, nf), h in res_ff.items()], size=2.6)

plt.figure(figsize=(6, 3))                                         # exercise 3
for k, sigma in enumerate([3, 30]):
    plt.semilogy(res_ff[sigma, 64]["train"], color=f"C{k}", label=f"sigma = {sigma}: train")
    plt.semilogy(res_ff[sigma, 64]["val"], "--", color=f"C{k}", label=f"sigma = {sigma}: validation")
plt.xlabel("epoch"); plt.ylabel("MSE"); plt.title("Small vs large sigma (n_freq = 64)"); plt.legend(fontsize=8); plt.show()

* Step 1: $\sigma = 3$ is the best (about 26 dB); $\sigma = 1$ is too smooth (about 25 dB); $\sigma = 10$ and $30$ overfit (about 16 and 12 dB, noisy images).
* Step 2: with $\sigma = 3$, more frequencies help: `n_freq = 16` gives about 24 dB, `n_freq = 256` about 28 dB. Best setting: $\sigma = 3$, `n_freq = 256`; its test PSNR is close to its validation PSNR. (With a long training, the best $\sigma$ is a little smaller, 1.5 to 2: part 2.)
* **Exercise 3:** with $\sigma = 30$ the training MSE keeps decreasing, but the validation MSE does not improve and ends about 15 times higher: overfitting from the first epochs. With $\sigma = 3$ the two curves stay close.

### Exercise 4

Same model as $\sigma = 3$, `n_freq = 64` (step 1), with $B$ trained with the other weights (`trainable=True`); we compare the validation PSNR:

In [ ]:
torch.manual_seed(0)
hist_B = fit_image(ff_model(64, 3, [128, 128], trainable=True), img, lr=1e-3, n_epochs=20, n_batch=100, title="trainable B, sigma = 3, n_freq = 64")
print(f"fixed B: {res_ff[3, 64]['psnr_val']:.1f} dB | trainable B: {hist_B['psnr_val']:.1f} dB")

About the same PSNR: the random frequencies drawn with a good $\sigma$ are already well suited, and the MLP does most of the work. The choice of $\sigma$ matters much more than training $B$.

### Exercise 5

See part 2, exercises 3 (color image) and 4 (your own image).